# Phase 3 — Confirmatory B0 - seed 44

Runs the existing sequential no-replay baseline on the same five fixed XNLI orders with one additional training seed (44). The notebook verifies that newly prepared train/test examples have the exact same data-manifest hash as the original Phase 2 B0 run before training. This produces 5 runs total and does not change model, prompt, training settings, or data selection.

## Before running

1. Enable Kaggle Internet and a GPU.
2. Accept the gated `meta-llama/Llama-3.2-1B` terms and add a Kaggle secret named `HF_TOKEN`.
3. Run all cells. The notebook saves a small ZIP with result JSONs, protocol manifests, and a summary; it does not package model checkpoints or XNLI text.

This is a confirmatory follow-up to the exploratory five-order, seed-42 RQ1 analysis. It does not by itself establish a winning prior; the RQ1 analysis should be rerun using the original plus these additional B0 result records.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

if not (REPO_ROOT / "phase2" / "run_baseline.py").is_file():
    raise FileNotFoundError("The cloned repository is missing the Phase 2 B0 runner.")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    str(REPO_ROOT / "phase2" / "requirements-phase2.txt"),
], check=True)
sys.path.insert(0, str(REPO_ROOT))

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

from phase2.preflight import collect_preflight
preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
BASELINE_MANIFEST_PATH = REPO_ROOT / "phase2" / "runs" / "b0_kaggle_2026-11-11" / "protocol_manifest.json"
if not BASELINE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Original B0 protocol manifest not found: {BASELINE_MANIFEST_PATH}")
BASELINE_MANIFEST = json.loads(BASELINE_MANIFEST_PATH.read_text(encoding="utf-8"))
BASE_DATA_MANIFEST_SHA256 = BASELINE_MANIFEST["data_manifest_sha256"]
RUN_ROOT = Path("/kaggle/working/phase3_confirmatory_b0_seed_44")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({"repo": str(REPO_ROOT), "run_root": str(RUN_ROOT), "baseline_data_manifest_sha256": BASE_DATA_MANIFEST_SHA256})

Cloning into '/kaggle/working/asymmetric-interference-continual-learning'...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

{
  "cuda_available": true,
  "cuda_device": "Tesla T4",
  "cuda_device_count": 2,
  "cuda_total_memory_gib": 14.56,
  "gated_model_metadata_checked": true,
  "huggingface_token_present": true,
  "model_id": "meta-llama/Llama-3.2-1B",
  "package_versions": {
    "accelerate": "1.15.0",
    "bitsandbytes": "0.50.2",
    "datasets": "5.0.1",
    "peft": "0.21.2",
    "torch": "2.11.0+cu130",
    "transformers": "5.18.0"
  },
  "python_version": "3.13.16",
  "tokenizer_pad_token_present": false
}
{'repo': '/kaggle/working/asymmetric-interference-continual-learning', 'run_root': '/kaggle/working/phase3_confirmatory_b0_seed_44', 'baseline_data_manifest_sha256': 'e459305688bf0ff5db4f744cee773ef2c0bb55c52df4bf71a7980479e58301a4'}


In [2]:
import copy
import shutil
import statistics
from zipfile import ZipFile, ZIP_DEFLATED

from phase2.prepare_data import prepare_xnli
from phase2.protocol import build_manifest, load_protocol, validate_protocol
from phase2.results import validate_result_record
from phase2.run_baseline import validate_prepared_artifacts

SEEDS = (44,)
BASE_PROTOCOL = load_protocol(PROTOCOL_PATH)
ORDER_IDS = [order["id"] for order in BASE_PROTOCOL["orders"]]
if len(ORDER_IDS) != 5:
    raise ValueError(f"Expected the five frozen stream orders, found {len(ORDER_IDS)}.")

EXPORT_ROOT = RUN_ROOT / "export"
for folder in ("results", "manifests", "protocols"):
    (EXPORT_ROOT / folder).mkdir(parents=True, exist_ok=True)
records_by_seed = {}
protocol_hashes = {}

for seed in SEEDS:
    protocol = copy.deepcopy(BASE_PROTOCOL)
    protocol["protocol_id"] = f"{BASE_PROTOCOL['protocol_id']}-confirmatory-seed-{seed}"
    protocol["training"]["seeds"] = [seed]
    validate_protocol(protocol)
    protocol_path = RUN_ROOT / "protocols" / f"protocol_seed_{seed}.json"
    protocol_path.parent.mkdir(parents=True, exist_ok=True)
    protocol_path.write_text(json.dumps(protocol, indent=2, sort_keys=True) + "\n", encoding="utf-8")

    artifact_root = RUN_ROOT / f"seed_{seed}"
    if not (artifact_root / "protocol_manifest.json").is_file():
        if (artifact_root / "data").exists():
            raise FileExistsError(f"Partial data folder without a manifest: {artifact_root / 'data'}")
        prepared = prepare_xnli(protocol_path, artifact_root)
    else:
        prepared = validate_prepared_artifacts(protocol, artifact_root)
    if prepared["data_manifest_sha256"] != BASE_DATA_MANIFEST_SHA256:
        raise ValueError(
            f"Seed {seed} selected XNLI data differs from the original B0 run. "
            "Training stopped to preserve the fixed-data comparison."
        )

    command = [
        sys.executable, "-m", "phase2.run_baseline",
        "--method", "B0",
        "--protocol", str(protocol_path),
        "--artifact-root", str(artifact_root),
    ]
    for order_id in ORDER_IDS:
        command.extend(["--order-id", order_id])
    print(f"Running B0 seed {seed} across {ORDER_IDS}")
    subprocess.run(command, cwd=REPO_ROOT, check=True)

    run_manifest = build_manifest(protocol)
    run_manifest["data_artifacts"] = prepared["data_artifacts"]
    run_manifest["data_manifest_sha256"] = prepared["data_manifest_sha256"]
    expected_runs = [run for run in run_manifest["runs"] if run["method_id"] == "B0"]
    if len(expected_runs) != len(ORDER_IDS):
        raise ValueError(f"Expected five B0 runs for seed {seed}; found {len(expected_runs)}.")

    seed_records = []
    for run in expected_runs:
        result_path = artifact_root / "results" / f"{run['run_id']}.json"
        if not result_path.is_file():
            raise FileNotFoundError(f"Missing completed B0 result: {result_path}")
        record = json.loads(result_path.read_text(encoding="utf-8"))
        validate_result_record(record, run_manifest)
        seed_records.append(record)
        shutil.copy2(result_path, EXPORT_ROOT / "results" / result_path.name)

    records_by_seed[seed] = seed_records
    protocol_hashes[seed] = run_manifest["protocol_sha256"]
    shutil.copy2(artifact_root / "protocol_manifest.json", EXPORT_ROOT / "manifests" / f"seed_{seed}_protocol_manifest.json")
    shutil.copy2(protocol_path, EXPORT_ROOT / "protocols" / protocol_path.name)

all_records = [record for seed in SEEDS for record in records_by_seed[seed]]
metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "purpose": "confirmatory additional B0 seeds for Phase 3 RQ1",
    "base_protocol_id": BASE_PROTOCOL["protocol_id"],
    "additional_training_seeds": list(SEEDS),
    "orders": ORDER_IDS,
    "completed_runs": len(all_records),
    "expected_runs": len(SEEDS) * len(ORDER_IDS),
    "original_data_manifest_sha256": BASE_DATA_MANIFEST_SHA256,
    "protocol_sha256_by_seed": {str(seed): protocol_hashes[seed] for seed in SEEDS},
    "per_seed_metrics": {
        str(seed): {
            name: {
                "mean": statistics.mean(record["metrics"][name] for record in records_by_seed[seed]),
                "population_std": statistics.pstdev(record["metrics"][name] for record in records_by_seed[seed]),
            }
            for name in metric_names
        }
        for seed in SEEDS
    },
    "all_additional_runs_metrics": {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in all_records),
            "population_std": statistics.pstdev(record["metrics"][name] for record in all_records),
        }
        for name in metric_names
    },
}
(EXPORT_ROOT / "confirmatory_b0_summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)

ZIP_PATH = Path("/kaggle/working/phase3_confirmatory_b0_seed_44_results.zip")
with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(EXPORT_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(EXPORT_ROOT).as_posix())
print(json.dumps(summary, indent=2, sort_keys=True))
print("Download and return:", ZIP_PATH)

README.md:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 50.2MB            

en/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  157kB            

... (87 outputs left out)

100%|██████████| 32/32 [02:23<00:00,  4.50s/it]


{'loss': '0.9937', 'grad_norm': '3.649', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.7423', 'grad_norm': '9.36', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.8006', 'grad_norm': '6.845', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '143.9', 'train_samples_per_second': '3.474', 'train_steps_per_second': '0.222', 'train_loss': '0.8223', 'epoch': '1'}


100%|██████████| 32/32 [02:44<00:00,  5.14s/it]


{'loss': '1.036', 'grad_norm': '4.301', 'learning_rate': '0.0001437', 'epoch': '0.32'}
{'loss': '0.8495', 'grad_norm': '6.954', 'learning_rate': '8.125e-05', 'epoch': '0.64'}
{'loss': '0.7337', 'grad_norm': '5.022', 'learning_rate': '1.875e-05', 'epoch': '0.96'}
{'train_runtime': '164.6', 'train_samples_per_second': '3.038', 'train_steps_per_second': '0.194', 'train_loss': '0.8511', 'epoch': '1'}
Completed xnli-pilot-5lang-500x200-v1-confirmatory-seed-44-B0-order-5-seed-44: ACC=0.5330
{
  "additional_training_seeds": [
    44
  ],
  "all_additional_runs_metrics": {
    "ACC": {
      "mean": 0.5346,
      "population_std": 0.018028865743579075
    },
    "BWT": {
      "mean": 0.06875,
      "population_std": 0.042463219378657574
    },
    "F": {
      "mean": 0.0024999999999999883,
      "population_std": 0.05721669336828194
    },
    "FWT": {
      "mean": 0.10450000000000001,
      "population_std": 0.018193405398660246
    }
  },
  "base_protocol_id": "xnli-pilot-5lang-500x200-v1

When complete, download `/kaggle/working/phase3_confirmatory_b0_seed_44_results.zip` from the Kaggle output files and provide it for the combined RQ1 analysis. The notebook includes the 5 validated result JSONs, per-seed protocols/manifests, and summary; prepared XNLI sentence files and model checkpoints remain outside the ZIP.